<p align="center">
  <img src="https://i.imgur.com/a3uAqnb.png" alt="KAUST Academy banner" width="100%">
</p>

# Inference Optimisation


## Inference Optimisation for Language Models

This lab tests three deployment ideas separately:

1. **KV cache** - what changes when we store previous attention keys and values?
2. **Quantization** - what changes when we load the same LLM in 4-bit instead of half precision?
3. **Knowledge distillation** - can a small attention-based classifier imitate a fine-tuned DistilBERT teacher?

For each part we measure something visible: speed, memory, answer quality, accuracy, or model size.


In [ ]:
from IPython.display import clear_output

!pip install -q -U datasets transformers accelerate "bitsandbytes>=0.46.1"

clear_output()
print("Packages installed.")


In [ ]:
import gc
import os
import math
import html
import random
import re
import string
import time
import warnings
from collections import Counter

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
)

warnings.filterwarnings("ignore")

# seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"Device : {DEVICE}")


### Small Timing and Memory Helpers

CUDA runs asynchronously, so timing a GPU operation without synchronizing can under-report the real time. The helper below waits for GPU work to finish before reading the clock.

The memory helpers are only used around the quantization experiment, where memory is one of the main results.


In [ ]:

def sync_if_cuda():
    '''
    Synchronize CUDA if available. This is useful for accurate timing of GPU operations.
    '''
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def clear_gpu_memory():
    ''' to clear GPU memory. we need it before measuring memory usage. '''
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def current_gpu_memory_mb():
    '''
    The Current GPU memory in MB, need it to measure memory usage.
    '''
    if not torch.cuda.is_available():
        return None
    return torch.cuda.memory_allocated() / 1e6


def peak_gpu_memory_mb():
    '''
    The Peak GPU memory in MB, need it to measure memory usage.
    '''
    if not torch.cuda.is_available():
        return None
    return torch.cuda.max_memory_allocated() / 1e6 # divide by 1e6 to convert bytes to MB


def count_parameters(model):
    '''
    Count the number of parameters in a model. just to comapre models complexity / capacity.
    '''
    return sum(p.numel() for p in model.parameters())


def tensor_memory_mb(tensor):
    '''
    Calculate the memory usage of a tensor in MB.
    '''
    if tensor is None:
        return 0.0
    return tensor.numel() * tensor.element_size() / 1e6


def model_storage_memory_mb(model):
    '''
    Calculate the memory usage of a model in MB. This includes the parameters and buffers.
    We need it to measure memory usage of the whole model.
    '''
    total = 0.0
    for param in model.parameters():
        total += tensor_memory_mb(param)
    for buffer in model.buffers():
        total += tensor_memory_mb(buffer)
    return total


def past_key_values_memory_mb(past_key_values):
    '''
    Calculate the memory usage of past_key_values in MB. This is useful for measuring the memory usage of the attention mechanism.
    '''
    total = 0.0
    for layer_cache in past_key_values:
        for tensor in layer_cache:
            total += tensor_memory_mb(tensor)
    return total


def load_squad_split(split):
    '''loading the SQuAD dataset split. This is useful for testing the model on different splits of the dataset.
    we used same data before :) '''

    return load_dataset("rajpurkar/squad", split=split)


# Part 1 - KV Cache

In a decoder-only LLM, generation has two phases:

- **Prefill:** process the whole prompt once and build the first KV cache.
- **Decode:** generate one new token at a time.

Without a KV cache, each decode step recomputes attention over the whole growing sequence. With a KV cache, the model reuses previous keys and values and only computes the new token's keys and values.

For one layer, after prefill we store the keys and values for the prompt tokens:

$$
K_{\text{cache}} \in \mathbb{R}^{B \times n_{kv} \times S \times d_{\text{head}}}
$$

$$
V_{\text{cache}} \in \mathbb{R}^{B \times n_{kv} \times S \times d_{\text{head}}}
$$

Where:

$$
B = \text{batch size}
$$

$$
S = \text{prompt sequence length}
$$

$$
n_{kv} = \text{number of key/value heads}
$$

$$
d_{\text{head}} = \text{dimension of each attention head}
$$

At the next decode step, the new token creates a query:

$$
Q_{\text{new}} \in \mathbb{R}^{B \times n_h \times 1 \times d_{\text{head}}}
$$

Where:

$$
n_h = \text{number of query attention heads}
$$

Then attention uses the cached keys and values:

$$
\text{scores}
=
Q_{\text{new}} K_{\text{cache}}^{T}
$$

Shape-wise:

$$
(B, n_h, 1, d_{\text{head}})
\times
(B, n_{kv}, d_{\text{head}}, S)
\rightarrow
(B, n_h, 1, S)
$$

Then we apply scaled softmax:

$$
\text{weights}
=
\operatorname{softmax}
\left(
\frac{\text{scores}}{\sqrt{d_{\text{head}}}}
\right)
$$

Finally, we get the context vector:

$$
\text{context}
=
\text{weights} \; V_{\text{cache}}
$$

Shape-wise:

$$
(B, n_h, 1, S)
\times
(B, n_{kv}, S, d_{\text{head}})
\rightarrow
(B, n_h, 1, d_{\text{head}})
$$

The memory cost of the cache is approximately:

$$
\text{KV cache bytes}
=
2
\times L
\times B
\times n_{kv}
\times S
\times d_{\text{head}}
\times \text{bytes per value}
$$

Where:

$$
L = \text{number of transformer layers}
$$

The factor of $2$ is because we store both **K** and **V**.

Here we use real QA prompts from **SQuAD**, then extend the context to controlled token lengths. This gives the experiment a realistic input format while still letting us isolate the effect of sequence length.

![KV cache computation with caching](https://miro.medium.com/v2/resize:fit:640/format:webp/0*_Opn4ZhUXMqfs22Y.png)

## Load the LLM and QA Examples

We load a small Qwen model and a small SQuAD validation slice. This cell also runs one warm-up forward pass before timing.

What we will do:
1. chooses the model and benchmark settings
2. defines how to load the model
3. defines the prompt format
4. loads SQuAD + model + warm-up run

In [ ]:

KV_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct" # Decoder only model
KV_TARGET_PROMPT_LENGTHS = [128, 512, 1024, 2048] # we will test on these lenthgs; however, KV cache becomes more useful when the prompt is long.
KV_BENCHMARK_POOL = 500 # Number of samples to use for benchmarking
KV_MAX_NEW_TOKENS = 64 # Maximum number of new tokens to generate
KV_REPEATS = 5 # Number of times to repeat each benchmark test. Why???


def load_half_precision_llm(model_id):
    '''
    Load a half-precision LLM model and tokenizer.
    '''
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    model.eval()
    return tokenizer, model


def format_squad_prompt(example, context=None):

    context = example["context"] if context is None else context
    return (
        "Answer the question using only the context. Keep the answer short.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {example['question']}\n"
        "Answer:"
    )


# We use real SQuAD text, then extend the context only to control prompt length.
squad_pool = load_squad_split(f"validation[:{KV_BENCHMARK_POOL}]")

clear_gpu_memory() # clear GPU memory before loading the model to get accurate memory usage measurements.
kv_tokenizer, kv_model = load_half_precision_llm(KV_MODEL_ID)

# One small warm-up keeps kernel setup and memory allocation out of the benchmark.
warmup_ids = kv_tokenizer(format_squad_prompt(squad_pool[0]), return_tensors="pt").input_ids.to(kv_model.device)
with torch.no_grad():
    _ = kv_model(input_ids=warmup_ids[:, :64], use_cache=True)
sync_if_cuda()

print("Loaded:", KV_MODEL_ID)
print("Current GPU memory (MB):", current_gpu_memory_mb())


## KV Cache Benchmark Functions

These helpers prepare SQuAD prompts, select examples by length, and run generation with and without cached keys/values.


In [ ]:

def make_prompt_at_length(example, tokenizer, target_tokens):
    '''
    Create a prompt for a SQuAD example that is at least target_tokens long.
    '''
    context_piece = example["context"].strip()
    context = context_piece

    # We repeat just to get a controlled prompt length. This is not for QA quality.
    for _ in range(30):
        prompt = format_squad_prompt(example, context=context)
        encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=target_tokens)
        if encoded.input_ids.shape[1] >= target_tokens:
            return prompt, encoded.input_ids, encoded.input_ids.shape[1]
        context = context + "\n\n" + context_piece

    return prompt, encoded.input_ids, encoded.input_ids.shape[1]


def build_controlled_squad_prompts(dataset, tokenizer, target_lengths):
    '''
    one example for each target length
    '''
    examples = []
    for target, example in zip(target_lengths, dataset):
        prompt, input_ids, prompt_tokens = make_prompt_at_length(example, tokenizer, target)
        examples.append({
            "target_tokens": target,
            "prompt": prompt,
            "input_ids": input_ids,
            "prompt_tokens": prompt_tokens,
            "question": example["question"],
            "gold_answers": example["answers"]["text"],
        })
    return examples


def estimate_kv_cache_mb(model, seq_len, dtype_bytes=2, batch_size=1):
    '''
    Estimate the memory usage of the KV cache in MB.
    '''
    config = model.config
    n_layers = config.num_hidden_layers
    n_heads = config.num_attention_heads
    n_kv_heads = getattr(config, "num_key_value_heads", n_heads)
    head_dim = config.hidden_size // n_heads # each head dimension is the total hidden size divided by number of heads
    total_bytes = batch_size * 2 * n_layers * n_kv_heads * seq_len * head_dim * dtype_bytes
    return total_bytes / 1e6 # convert bytes to MB


def generate_with_cache_once(model, input_ids, max_new_tokens):
    '''
    Generate text using the model WITH KV cache ENABLED.
    '''
    with torch.no_grad():
        sync_if_cuda()
        t0 = time.perf_counter() # Measure the time for the prefill step
        out = model(input_ids=input_ids, use_cache=True) # NOTICE use_cache=True
        sync_if_cuda()
        prefill_ms = (time.perf_counter() - t0) * 1000 # convert seconds to milliseconds

        past = out.past_key_values # NOTICE we are storing the past_key_values for the next decode steps
        cache_mb = past_key_values_memory_mb(past)
        next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True) # B,S,Vocab --> [:, -1, :] takes the last token's logits
        generated = [next_token]
        decode_times = []

        # The first generated token came from prefill. These are the post-prefill decode steps.
        for _ in range(max_new_tokens - 1):
            sync_if_cuda()
            t0 = time.perf_counter()
            out = model(input_ids=next_token, past_key_values=past, use_cache=True) # NOTICE we are passing the past_key_values
            sync_if_cuda()
            decode_times.append((time.perf_counter() - t0) * 1000)

            past = out.past_key_values
            next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated.append(next_token)

    return prefill_ms, decode_times, torch.cat(generated, dim=1), cache_mb


def generate_without_cache_once(model, input_ids, max_new_tokens):
    '''
    Generate text using the model WITHOUT KV cache .
    '''
    with torch.no_grad():
        sync_if_cuda()
        t0 = time.perf_counter()
        out = model(input_ids=input_ids, use_cache=False) # NOTICE use_cache=False
        sync_if_cuda()
        prefill_ms = (time.perf_counter() - t0) * 1000 # convert seconds to milliseconds

        next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        seq = torch.cat([input_ids, next_token], dim=1) ### WHAT IS SEQ, WHY NOT JUST NEXT_TOKEN??????
        generated = [next_token]
        decode_times = []

        # Same number of post-prefill decode steps as the cached path.
        for _ in range(max_new_tokens - 1):
            sync_if_cuda()
            t0 = time.perf_counter()
            out = model(input_ids=seq, use_cache=False)
            sync_if_cuda()
            decode_times.append((time.perf_counter() - t0) * 1000)

            next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated.append(next_token)
            seq = torch.cat([seq, next_token], dim=1)

    return prefill_ms, decode_times, torch.cat(generated, dim=1)


def benchmark_kv_example(model, tokenizer, input_ids, max_new_tokens, repeats):
    '''
    Benchmark the model with and without KV cache for a single example.'''
    cached_prefill, cached_decode, no_cache_decode = [], [], []
    cached_ids = no_cache_ids = None
    cache_mb = None

    for _ in range(repeats):
        prefill_ms, decode_times, cached_ids, cache_mb = generate_with_cache_once(model, input_ids, max_new_tokens)
        cached_prefill.append(prefill_ms)
        cached_decode.append(np.mean(decode_times))

        # we dont need the prefill time for the no-cache path, since it is the same as the cached path.
        # We only need the decode times.
        _, decode_times, no_cache_ids = generate_without_cache_once(model, input_ids, max_new_tokens)
        no_cache_decode.append(np.mean(decode_times))

    return {
        "prefill_mean": float(np.mean(cached_prefill)),
        "cached_decode_mean": float(np.mean(cached_decode)),
        "no_cache_decode_mean": float(np.mean(no_cache_decode)),
        "cached_ids": cached_ids,
        "no_cache_ids": no_cache_ids,
        "cache_mb": cache_mb,
    }

## NOTE:The reported cache memory is the cache after processing the prompt. During generation it grows slightly as each new token is added.


## Run the KV Cache Benchmark

For each selected prompt length, we compare cached decoding against recomputing the full sequence.


In [ ]:

kv_examples = build_controlled_squad_prompts(squad_pool, kv_tokenizer, KV_TARGET_PROMPT_LENGTHS)

kv_rows = []

for sample in kv_examples:
    input_ids = sample["input_ids"].to(kv_model.device)
    result = benchmark_kv_example(
        kv_model,
        kv_tokenizer,
        input_ids,
        max_new_tokens=KV_MAX_NEW_TOKENS,
        repeats=KV_REPEATS,
    )

    kv_rows.append({
        "target_tokens": sample["target_tokens"],
        "prompt_tokens": sample["prompt_tokens"],
        "prefill_ms_mean": round(result["prefill_mean"], 2),
        "with_cache_ms/token": round(result["cached_decode_mean"], 2),
        "without_cache_ms/token": round(result["no_cache_decode_mean"], 2),
        "speedup": round(result["no_cache_decode_mean"] / max(result["cached_decode_mean"], 1e-8), 2),
        "cache_tensor_mb": round(result["cache_mb"], 2),
        "cache_estimate_mb": round(estimate_kv_cache_mb(kv_model, sample["prompt_tokens"]), 2),
        "question": sample["question"][:70],
    })

kv_df = pd.DataFrame(kv_rows)
kv_df


**Interpretation**

As the prompt becomes longer, decoding **without KV cache** becomes much slower because the model recomputes attention over the full growing sequence at every new token. In this run, the no-cache decode time increased from about `76.30 ms/token` at `128` prompt tokens to about `611.61 ms/token` at `2048` prompt tokens.

With KV cache, decode time stays much lower because the model reuses the stored keys and values from the prompt. The benefit becomes much clearer for long prompts: at `2048` prompt tokens, cached decoding was about `17.58x` faster.

The cache memory also grows with sequence length: from `1.57 MB` at `128` tokens to `25.17 MB` at `2048` tokens. 

```text
KV cache = much faster decoding, but extra memory for stored keys and values.

## Plot the Average Decode Time

The table gives exact numbers; the plot makes the trend easier to see.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    kv_df["prompt_tokens"],
    kv_df["with_cache_ms/token"],
    marker="o",
    label="with KV cache",
)
plt.plot(
    kv_df["prompt_tokens"],
    kv_df["without_cache_ms/token"],
    marker="o",
    label="without KV cache",
)
plt.xlabel("Prompt length (tokens)")
plt.ylabel("Decode time per token (ms)")
plt.title(f"KV cache timing, averaged over {KV_REPEATS} runs")
plt.legend()
plt.tight_layout()
plt.show()


## Inspect Decode Latency Step by Step

This plot uses the longest selected prompt to show how decode latency changes across generation steps.


In [ ]:
longest_sample = max(kv_examples, key=lambda item: item["prompt_tokens"])
longest_ids = longest_sample["input_ids"].to(kv_model.device)

_, cached_steps, _, _ = generate_with_cache_once(kv_model, longest_ids, KV_MAX_NEW_TOKENS)
_, no_cache_steps, _ = generate_without_cache_once(kv_model, longest_ids, KV_MAX_NEW_TOKENS)

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cached_steps) + 1), cached_steps, marker="o", label="with KV cache")
plt.plot(range(1, len(no_cache_steps) + 1), no_cache_steps, marker="o", label="without KV cache")
plt.xlabel("Post-prefill decode step")
plt.ylabel("Step latency (ms)")
plt.title(f"Per-step decode latency for {longest_sample['prompt_tokens']} prompt tokens")
plt.legend()
plt.tight_layout()
plt.show()


**What to notice**

The KV cache does **not** remove the prefill cost. The full prompt is still processed once to build the initial key/value cache.

The gain appears during decode. With KV cache, each step only processes the newly generated token and attends to the stored keys and values. Without KV cache, every step recomputes the whole growing prefix again.

This is why the per-step latency chart has a large gap: for the `2048`-token prompt, cached decoding stays around `35-50 ms` per token, while decoding without cache stays around `600+ ms` per token.


# Part 2 - Quantization

Quantization stores model weights with fewer bits. In this section we load the same LLM twice:

1. normal half precision
2. 4-bit NF4 using `bitsandbytes`

The basic idea is to replace high-precision weights with low-precision codes plus a scale.

For a simple symmetric quantizer, one weight $w$ is converted into a low-bit code $q$:

$$
q = \operatorname{clip}\left(\operatorname{round}\left(\frac{w}{s}\right), q_{\min}, q_{\max}\right)
$$

During computation, the stored code is approximately converted back:

$$
\hat{w} = s \cdot q
$$

Where:

$$
w = \text{original weight}, \quad
s = \text{scale}, \quad
q = \text{stored low-bit code}, \quad
\hat{w} = \text{reconstructed approximate weight}
$$

Example:

$$
w = 0.73, \quad s = 0.10
$$

$$
q = \operatorname{round}\left(\frac{0.73}{0.10}\right) = 7
$$

$$
\hat{w} = 0.10 \times 7 = 0.70
$$

So the value is not stored perfectly, but it is close enough for many neural network weights.

The memory saving is the main point. If a model has $N$ weights:

$$
\text{FP16 memory} = N \times 16 \text{ bits}
$$

$$
\text{4-bit memory} = N \times 4 \text{ bits}
$$

So, ideally:

$$
\frac{\text{4-bit memory}}{\text{FP16 memory}} = \frac{4}{16} = \frac{1}{4}
$$

NF4 is a special 4-bit format commonly used for neural network weights. It is designed around the fact that many trained weights are concentrated near zero, so its levels are not just equally spaced integers.

Then we compare GPU memory, generation speed, and answer quality on a small SQuAD subset.

Note the example and formulas mentioned above is the simple INT4, NF4 is a smarter 4-bit variant.


## Quantization Evaluation Helpers

These functions generate SQuAD answers and calculate a light token-overlap quality score.


In [ ]:

QUANT_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
QUANT_EVAL_SIZE = 200
QUANT_MAX_NEW_TOKENS = 32



def normalize_english_text(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip().lower()


def calculate_token_f1(predicted, ground_truth):
    pred_tokens = normalize_english_text(predicted).split()
    truth_tokens = normalize_english_text(ground_truth).split()

    if len(pred_tokens) == 0 and len(truth_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 0.0

    common_tokens = Counter(pred_tokens) & Counter(truth_tokens)
    num_common = sum(common_tokens.values())
    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall = num_common / len(truth_tokens)
    return 2 * (precision * recall) / (precision + recall)


def generate_answer(model, tokenizer, prompt, max_new_tokens):
    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_tokens = inputs["input_ids"].shape[1]

        sync_if_cuda()
        t0 = time.perf_counter()
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        sync_if_cuda()
        elapsed = time.perf_counter() - t0

        new_tokens = out.shape[1] - input_tokens
        answer = tokenizer.decode(out[0, input_tokens:], skip_special_tokens=True).strip()

    return answer, new_tokens / max(elapsed, 1e-8)


def warmup_generation(model, tokenizer, example):
    prompt = format_squad_prompt(example)
    with torch.no_grad():
        _ = generate_answer(model, tokenizer, prompt, max_new_tokens=4)
    sync_if_cuda()


def evaluate_squad_subset(model, tokenizer, examples, max_new_tokens):
    rows = []
    tokens_per_second = []

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for example in tqdm(examples, desc="QA evaluation"):
        prompt = format_squad_prompt(example)
        answer, tps = generate_answer(model, tokenizer, prompt, max_new_tokens)
        gold_answers = example["answers"]["text"]
        best_f1 = max(calculate_token_f1(answer, gold) for gold in gold_answers)
        contains_gold = any(normalize_english_text(gold) in normalize_english_text(answer) for gold in gold_answers)

        rows.append({
            "question": example["question"],
            "gold": gold_answers[0],
            "answer": answer,
            "best_f1": best_f1,
            "contains_gold": contains_gold,
        })
        tokens_per_second.append(tps)

    return {
        "rows": rows,
        "avg_f1": float(np.mean([row["best_f1"] for row in rows])),
        "contains_gold_rate": float(np.mean([row["contains_gold"] for row in rows])),
        "tokens_per_second": float(np.mean(tokens_per_second)),
        "peak_mb": peak_gpu_memory_mb(),
    }


def load_4bit_llm(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # NF4 is the common 4-bit format used in QLoRA-style loading.
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model


quant_eval_examples = list(load_squad_split(f"validation[:{QUANT_EVAL_SIZE}]"))


## Compare FP16 and 4-bit Loading

We load the model in FP16 first, then 4-bit NF4, and compare memory, speed, and quick QA quality.


In [ ]:

quant_rows = []
quant_sample_rows = []

if "kv_model" in globals():
    del kv_model
clear_gpu_memory()

fp_before_load = current_gpu_memory_mb()
fp_tokenizer, fp_model = load_half_precision_llm(QUANT_MODEL_ID)
fp_after_load = current_gpu_memory_mb()
fp_storage_mb = model_storage_memory_mb(fp_model)
warmup_generation(fp_model, fp_tokenizer, quant_eval_examples[0])
fp_eval = evaluate_squad_subset(fp_model, fp_tokenizer, quant_eval_examples, QUANT_MAX_NEW_TOKENS)
quant_rows.append({
    "variant": "FP16" if DEVICE == "cuda" else "FP32 CPU",
    "gpu_mb_before_load": None if fp_before_load is None else round(fp_before_load, 1),
    "gpu_mb_after_load": None if fp_after_load is None else round(fp_after_load, 1),
    "model_storage_mb": round(fp_storage_mb, 1),
    "peak_inference_mb": None if fp_eval["peak_mb"] is None else round(fp_eval["peak_mb"], 1),
    "tokens/sec_mean": round(fp_eval["tokens_per_second"], 2),
    "avg_f1": round(fp_eval["avg_f1"], 3),
    "contains_gold_rate": round(fp_eval["contains_gold_rate"], 3),
})

for row in fp_eval["rows"][:3]:
    quant_sample_rows.append({"variant": "FP16", **row})

del fp_model
clear_gpu_memory()

if DEVICE == "cuda":
    q_before_load = current_gpu_memory_mb()
    q_tokenizer, q_model = load_4bit_llm(QUANT_MODEL_ID)
    q_after_load = current_gpu_memory_mb()
    q_storage_mb = model_storage_memory_mb(q_model)
    warmup_generation(q_model, q_tokenizer, quant_eval_examples[0])
    q_eval = evaluate_squad_subset(q_model, q_tokenizer, quant_eval_examples, QUANT_MAX_NEW_TOKENS)
    quant_rows.append({
        "variant": "4-bit NF4",
        "gpu_mb_before_load": round(q_before_load, 1),
        "gpu_mb_after_load": round(q_after_load, 1),
        "model_storage_mb": round(q_storage_mb, 1),
        "peak_inference_mb": round(q_eval["peak_mb"], 1),
        "tokens/sec_mean": round(q_eval["tokens_per_second"], 2),
        "avg_f1": round(q_eval["avg_f1"], 3),
        "contains_gold_rate": round(q_eval["contains_gold_rate"], 3),
    })

    for row in q_eval["rows"][:3]:
        quant_sample_rows.append({"variant": "4-bit NF4", **row})

    del q_model
    clear_gpu_memory()
else:
    print("4-bit loading skipped because bitsandbytes quantization needs a CUDA GPU.")

quant_df = pd.DataFrame(quant_rows)
quant_samples_df = pd.DataFrame(quant_sample_rows)

quant_df


**Interpretation**

The main effect of 4-bit quantization is memory reduction. In this run, the FP16 model used about `998.4 MB` after loading, while the 4-bit NF4 model used about `483.6 MB`. So the quantized model used roughly half the GPU memory after loading.

`model_storage_mb` estimates how much memory the model parameters and buffers occupy. This also drops from about `988.1 MB` in FP16 to about `451.3 MB` in 4-bit NF4.

The 4-bit model was not faster here. FP16 generated about `27.12 tokens/sec`, while 4-bit NF4 generated about `17.33 tokens/sec`. This can happen because 4-bit weights may need dequantization during computation, and the GPU may be more optimized for FP16 operations.



## Inspect a Few Generated Answers

The summary table gives the metrics. This table lets us quickly see what the model actually said.


In [ ]:
quant_samples_df[["variant", "question", "gold", "answer", "best_f1", "contains_gold"]]


## Plot Quantization Memory

This chart shows the peak GPU memory measured during generation.


In [ ]:

if quant_df["gpu_mb_after_load"].notna().sum() > 1:
    x = np.arange(len(quant_df))
    width = 0.35

    plt.figure(figsize=(7, 4))
    plt.bar(x - width / 2, quant_df["gpu_mb_after_load"], width, label="after loading")
    plt.bar(x + width / 2, quant_df["peak_inference_mb"], width, label="peak during generation")
    plt.xticks(x, quant_df["variant"])
    plt.ylabel("GPU memory (MB)")
    plt.title("Quantization memory comparison")
    plt.legend()
    plt.tight_layout()
    plt.show()



# Part 3 - Knowledge Distillation on IMDB

For distillation we use real text data, because now we care about task accuracy.

The teacher is `distilbert-base-uncased`, fine-tuned on IMDB. The student is a tiny Transformer encoder classifier, close in spirit to the multi-head attention model from Week 6.

We compare:

- **Teacher:** larger transformer model
- **Distilled student:** smaller Transformer encoder trained with teacher soft targets and true labels

<!-- Add the KD diagram here when the image file/link is available. -->
<!-- Example: ![Knowledge distillation flow](path_or_url_to_kd_diagram.png) -->

## 1. Teacher and Student Outputs

For a batch of movie reviews, both models output logits:

$$
z^{(t)} \in \mathbb{R}^{B \times C}
$$

$$
z^{(s)} \in \mathbb{R}^{B \times C}
$$

Where:

$$
B = \text{batch size}, \qquad C = \text{number of classes}
$$

For IMDB sentiment classification:

$$
C = 2 \quad \text{negative / positive}
$$

Here:

$$
z^{(t)} = \text{teacher logits}, \qquad z^{(s)} = \text{student logits}
$$

## 2. Hard-Label Training

If we train only with the true label $y$, we use cross entropy:

$$
\mathcal{L}_{CE} = \operatorname{CE}(z^{(s)}, y)
$$

This teaches the student only the final correct class.

Example hard label:

$$
y = [0, 1]
$$

Meaning:

$$
\text{negative} = 0, \qquad \text{positive} = 1
$$

## 3. Soft Targets From the Teacher

The teacher probabilities are produced with softmax:

$$
p^{(t)} = \operatorname{softmax}(z^{(t)})
$$

The student probabilities are:

$$
p^{(s)} = \operatorname{softmax}(z^{(s)})
$$

Example soft teacher output:

$$
p^{(t)} = [0.28, 0.72]
$$

Meaning:

$$
\text{negative} = 0.28, \qquad \text{positive} = 0.72
$$

This tells the student: the review is positive, but not extremely obvious.

## 4. Distillation Loss

The distillation loss makes the student distribution match the teacher distribution:

$$
\mathcal{L}_{KD} = \operatorname{KL}\left(p^{(t)} \; || \; p^{(s)}\right)
$$

KL divergence measures how different the student probabilities are from the teacher probabilities.

## 5. Final Student Loss

The final loss combines both signals:

$$
\mathcal{L} = \alpha \mathcal{L}_{KD} + (1 - \alpha) \mathcal{L}_{CE}
$$

Where:

$$
\alpha = \text{how much we trust the teacher signal}
$$

So the student learns from:

1. the true label
2. the teacher's softer probability distribution

The goal is to get a smaller model that keeps much of the teacher's behavior while being cheaper to run.


## Prepare IMDB for Distillation

We use a small IMDB subset so the teacher and students can train in a reasonable time.


In [ ]:

TEACHER_MODEL_ID = "distilbert-base-uncased"
IMDB_TRAIN_SIZE = 25_000
IMDB_EVAL_SIZE = 5_000
MAX_TEXT_LEN = 256
TEACHER_BATCH_SIZE = 16
STUDENT_BATCH_SIZE = 64
TEACHER_EPOCHS = 3
STUDENT_EPOCHS = 5
DISTILL_ALPHA = 0.7

STUDENT_EMBED_DIM = 64
STUDENT_HEADS = 2
STUDENT_LAYERS = 1
STUDENT_FF_DIM = 128

# IMDB is familiar from previous labs and gives us a simple binary task.
imdb = load_dataset("stanfordnlp/imdb")
train_raw = imdb["train"].shuffle(seed=SEED).select(range(IMDB_TRAIN_SIZE))
eval_raw = imdb["test"].shuffle(seed=SEED).select(range(IMDB_EVAL_SIZE))

print(train_raw[0]["text"][:250], "...")
print("Label:", train_raw[0]["label"], "(0=negative, 1=positive)")


## Build the DistilBERT Teacher

The teacher is a pretrained DistilBERT classifier head that we fine-tune on IMDB.


In [ ]:
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
# The classification head starts fresh, then we fine-tune it below.
teacher_model = AutoModelForSequenceClassification.from_pretrained(
    TEACHER_MODEL_ID,
    num_labels=2,
).to(DEVICE)


def tokenize_for_teacher(batch):
    return teacher_tokenizer(batch["text"], truncation=True, max_length=MAX_TEXT_LEN)


train_teacher_ds = train_raw.map(tokenize_for_teacher, batched=True, remove_columns=["text"])
eval_teacher_ds = eval_raw.map(tokenize_for_teacher, batched=True, remove_columns=["text"])
train_teacher_ds.set_format("torch")
eval_teacher_ds.set_format("torch")

teacher_collator = DataCollatorWithPadding(tokenizer=teacher_tokenizer)
teacher_train_loader = DataLoader(
    train_teacher_ds,
    batch_size=TEACHER_BATCH_SIZE,
    shuffle=True,
    collate_fn=teacher_collator,
)
teacher_eval_loader = DataLoader(
    eval_teacher_ds,
    batch_size=TEACHER_BATCH_SIZE,
    shuffle=False,
    collate_fn=teacher_collator,
)

print("Teacher parameters:", f"{count_parameters(teacher_model):,}")


## Fine-tune and Evaluate the Teacher

This is the first training run in the distillation section.


In [ ]:
def move_batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}


def evaluate_teacher(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch)
            labels_tensor = batch.pop("labels") if "labels" in batch else batch.pop("label")
            logits = model(**batch).logits
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
            labels.extend(labels_tensor.cpu().tolist())
    return accuracy_score(labels, preds)


def train_teacher(model, loader, epochs):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    for epoch in range(epochs):
        losses = []
        progress = tqdm(loader, desc=f"Teacher epoch {epoch + 1}/{epochs}")
        for batch in progress:
            batch = move_batch_to_device(batch)
            loss = model(**batch).loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            progress.set_postfix(loss=f"{np.mean(losses[-20:]):.3f}")


train_teacher(teacher_model, teacher_train_loader, epochs=TEACHER_EPOCHS)
teacher_acc = evaluate_teacher(teacher_model, teacher_eval_loader)
print(f"Teacher IMDB accuracy: {teacher_acc:.3f}")


## Student Model

```text
Token ids -> token embeddings +  positional encoding -> Transformer encoder -> masked mean pooling -> classifier
```

It is much smaller than the teacher, so we expect cheaper inference. The question is how much accuracy it can keep after normal training and after distillation.


## Build the Student Vocabulary

The student uses the same lightweight text cleaning style from the earlier attention lab.


In [ ]:
STOP_WORDS = {
    "the", "a", "an", "and", "or", "but", "if", "in", "on", "at", "to", "of",
    "for", "with", "by", "as", "is", "was", "are", "were", "be", "been", "being",
    "it", "its", "this", "that", "these", "those", "so", "very", "i", "me", "my",
    "you", "your", "he", "she", "they", "them", "we", "us",
}

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_ID = 0
UNK_ID = 1
MAX_VOCAB = 20_000
STUDENT_MAX_LEN = 200

punct_pat = f"[{re.escape(string.punctuation)}]"
url_pat = r"https?://\S+|www\.\S+"
html_tag_pat = r"<[^>]+>"


def basic_tokenize(text, keep_stops=False):
    text = html.unescape(text)
    text = re.sub(html_tag_pat, " ", text)
    text = re.sub(url_pat, " ", text)
    text = text.lower()
    text = re.sub(punct_pat, " ", text)

    tokens = text.split()
    if not keep_stops:
        tokens = [tok for tok in tokens if tok not in STOP_WORDS]
    return tokens


counter = Counter()
for text in train_raw["text"]:
    counter.update(basic_tokenize(text))

vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for word, _ in counter.most_common(MAX_VOCAB - len(vocab)):
    vocab[word] = len(vocab)


def encode_student_text(text, max_len=STUDENT_MAX_LEN):
    ids = [vocab.get(tok, UNK_ID) for tok in basic_tokenize(text)[:max_len]]
    ids += [PAD_ID] * (max_len - len(ids))
    return ids


class IMDbStudentDataset(Dataset):
    def __init__(self, hf_split, soft_targets=None):
        self.texts = hf_split["text"]
        self.labels = hf_split["label"]
        self.soft_targets = soft_targets

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(encode_student_text(self.texts[idx]), dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }
        if self.soft_targets is not None:
            item["soft_target"] = self.soft_targets[idx]
        return item


print("Vocabulary size:", len(vocab))
print("Example ids:", encode_student_text(train_raw[0]["text"])[:20])


## Define the Mini Transformer Student

This student is encoder-only: it predicts one sentiment label, so it does not need decoder layers.


In [ ]:

class MiniTransformerStudent(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len=STUDENT_MAX_LEN,
        embed_dim=STUDENT_EMBED_DIM,
        num_heads=STUDENT_HEADS,
        num_layers=STUDENT_LAYERS,
        ff_dim=STUDENT_FF_DIM,
        num_classes=2,
    ):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.pos_embed = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=0.1,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids):
        mask = input_ids != PAD_ID
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)

        x = self.token_embed(input_ids) + self.pos_embed(positions)
        h = self.encoder(x, src_key_padding_mask=~mask)

        # Mean-pool only real tokens; padding should not vote in the sentence vector.
        mask_float = mask.unsqueeze(-1).float()
        pooled = (h * mask_float).sum(dim=1) / mask_float.sum(dim=1).clamp(min=1.0)
        logits = self.classifier(pooled)
        return logits, None


preview_student = MiniTransformerStudent(len(vocab))
print("Student parameters:", f"{count_parameters(preview_student):,}")
del preview_student


## Collect Teacher Soft Targets

This is inference, not training. The teacher produces softened probabilities for the student to learn from.


In [ ]:

def collect_teacher_soft_targets(model, loader):
    model.eval()
    soft_targets = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Collecting teacher probabilities"):
            batch = move_batch_to_device(batch)
            if "labels" in batch:
                batch.pop("labels")
            elif "label" in batch:
                batch.pop("label")

            logits = model(**batch).logits
            probs = F.softmax(logits, dim=-1).cpu()
            soft_targets.append(probs)
    return torch.cat(soft_targets, dim=0)


teacher_soft_loader = DataLoader(
    train_teacher_ds,
    batch_size=TEACHER_BATCH_SIZE,
    shuffle=False,
    collate_fn=teacher_collator,
)
soft_targets = collect_teacher_soft_targets(
    teacher_model,
    teacher_soft_loader,
)

print("Soft target shape:", tuple(soft_targets.shape))
print("First soft target:", soft_targets[0].numpy())


## Train the Distilled Student

We train one small student using both the teacher soft targets and the true labels.

In [ ]:

train_student_kd_ds = IMDbStudentDataset(train_raw, soft_targets=soft_targets)
eval_student_ds = IMDbStudentDataset(eval_raw)

student_kd_loader = DataLoader(train_student_kd_ds, batch_size=STUDENT_BATCH_SIZE, shuffle=True)
student_eval_loader = DataLoader(eval_student_ds, batch_size=STUDENT_BATCH_SIZE, shuffle=False)


def train_distilled_student(model, loader, epochs):
    model.to(DEVICE)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)

    for epoch in range(epochs):
        losses = []
        progress = tqdm(loader, desc=f"Student epoch {epoch + 1}/{epochs}")
        for batch in progress:
            input_ids = batch["input_ids"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            soft = batch["soft_target"].to(DEVICE)

            logits, _ = model(input_ids)
            ce_loss = F.cross_entropy(logits, labels)
            kd_loss = F.kl_div(
                F.log_softmax(logits, dim=-1),
                soft,
                reduction="batchmean",
            )
            loss = DISTILL_ALPHA * kd_loss + (1 - DISTILL_ALPHA) * ce_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            progress.set_postfix(loss=f"{np.mean(losses[-20:]):.3f}")


def evaluate_student(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            labels_tensor = batch["label"].to(DEVICE)
            logits, _ = model(input_ids)
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
            labels.extend(labels_tensor.cpu().tolist())
    return accuracy_score(labels, preds)


In [ ]:

student_kd = MiniTransformerStudent(len(vocab)).to(DEVICE)

train_distilled_student(student_kd, student_kd_loader, epochs=STUDENT_EPOCHS)
student_kd_acc = evaluate_student(student_kd, student_eval_loader)

print(f"Distilled student accuracy: {student_kd_acc:.3f}")


## Compare Accuracy, Size, and Inference Speed

The final table shows what the smaller student gains and what it loses compared with the teacher.


In [ ]:

def measure_classifier_latency(model, loader, model_type, n_batches=30):
    model.eval()
    times = []
    seen = 0
    iterator = iter(loader)

    # One unmeasured batch avoids counting first-call setup in the timing.
    try:
        warmup_batch = next(iterator)
        with torch.no_grad():
            if model_type == "teacher":
                warmup_batch = move_batch_to_device(warmup_batch)
                warmup_batch.pop("labels", None)
                warmup_batch.pop("label", None)
                model(**warmup_batch)
            else:
                model(warmup_batch["input_ids"].to(DEVICE))
        sync_if_cuda()
    except StopIteration:
        return {"ms/batch": 0.0, "examples/sec": 0.0}

    with torch.no_grad():
        for i, batch in enumerate(iterator):
            if i >= n_batches:
                break

            if model_type == "teacher":
                batch = move_batch_to_device(batch)
                batch.pop("labels", None)
                batch.pop("label", None)

                batch_size = next(iter(batch.values())).shape[0]
                sync_if_cuda()
                t0 = time.perf_counter()
                model(**batch)
                sync_if_cuda()
            else:
                input_ids = batch["input_ids"].to(DEVICE)
                batch_size = input_ids.shape[0]
                sync_if_cuda()
                t0 = time.perf_counter()
                model(input_ids)
                sync_if_cuda()

            times.append((time.perf_counter() - t0) * 1000)
            seen += batch_size

    total_seconds = sum(times) / 1000
    return {
        "ms/batch": float(np.mean(times)),
        "examples/sec": seen / max(total_seconds, 1e-8),
    }


teacher_latency = measure_classifier_latency(teacher_model, teacher_eval_loader, "teacher")
student_kd_latency = measure_classifier_latency(student_kd, student_eval_loader, "student")

summary_rows = []
for name, model, acc, latency in [
    ("DistilBERT teacher", teacher_model, teacher_acc, teacher_latency),
    ("Distilled student", student_kd, student_kd_acc, student_kd_latency),
]:
    summary_rows.append({
        "model": name,
        "parameters": f"{count_parameters(model):,}",
        "accuracy": round(acc, 3),
        "ms/batch_mean": round(latency["ms/batch"], 2),
        "examples/sec": round(latency["examples/sec"], 1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


**What to notice**

The teacher is much larger and more accurate, but the distilled student is far cheaper to run. In this run, the DistilBERT teacher has `66,955,010` parameters and reaches `90.5%` accuracy, while the distilled student has only `1,326,402` parameters and reaches `84.7%` accuracy.

So the student keeps a reasonable amount of the teacher's performance while using far fewer parameters:

```text
Teacher parameters   = 66,955,010
Student parameters   = 1,326,402
Student is about 50x smaller

# Final Summary

| Technique | Main idea | Compared to | What improves | Tradeoff | What we measured |
|---|---|---|---|---|---|
| KV cache | Reuse stored keys and values during decoding | Recomputing the full sequence at every decode step | Decode latency, especially for long prompts | Extra memory for cached K/V tensors | ms/token, speedup, cache memory |
| 4-bit quantization | Store model weights with fewer bits | FP16 loading | Model memory | Quality and speed can change, so we measure them | Load memory, peak inference memory, tokens/sec, QA F1 |
| Knowledge distillation | Train a small student using teacher probabilities | larger fine-tuned teacher | Model size and inference latency | Extra teacher-training/labeling step, possible accuracy drop | teacher vs student accuracy, parameter count, inference latency |

Inference optimisation is not one trick. In production we often combines several choices: a smaller model, quantized weights, KV caching ....


Contributed by: Hassan Mohammed Nasr 